# Step 1

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True, linewidth=120)

In [2]:
def phi(a,p):
    """Map field elements to roots of unity"""
    a = np.asarray(a)
    return np.exp(2j * np.pi * a / p)

In [9]:
p = 5
print("phi over GF({}) is = ".format(p))

for a in range(p): 
    print(f"phi({a}) = {phi(a,p): .3f}")

#Isomorphism check
a, b = 3, 4
print(f"\nphi({a} + {b}) = phi({(a+b)%p}) = {phi((a+b)%p,p): .3f}")
print(f"phi({a}) * phi({b}) = {phi(a,p) * phi(b,p): .3f}")


phi over GF(5) is = 
phi(0) =  1.000+0.000j
phi(1) =  0.309+0.951j
phi(2) = -0.809+0.588j
phi(3) = -0.809-0.588j
phi(4) =  0.309-0.951j

phi(3 + 4) = phi(2) = -0.809+0.588j
phi(3) * phi(4) = -0.809+0.588j


# Step 2 - The outer code: Reed Solomon

In [11]:
def reed_solomon_encode(message, p):
    """message: length-K coefficients over F_p. Returns length-N=p codeword"""
    message = np.asarray(message, dtype=int)
    points =  np.arange(p)
    powers = points[:, None] ** np.arange(len(message))
    return (powers @ message) % p

In [18]:
p = 5 
message = [2,1]
codeword = reed_solomon_encode(message, p)
print("message (coeffs):",message)
print("codeword f(0...4):",codeword, "<- [2 3 4 0 1] mod 5\n")


#some examples
for msg in [[2,1], [1,2], [3,4]]:
    print(f"   {msg} -> {reed_solomon_encode(msg, p)}")

message (coeffs): [2, 1]
codeword f(0...4): [2 3 4 0 1] <- [2 3 4 0 1] mod 5

   [2, 1] -> [2 3 4 0 1]
   [1, 2] -> [1 3 0 2 4]
   [3, 4] -> [3 2 1 0 4]


## point wise distance demo

In [34]:
from itertools import product

p, K = 5, 2
codewords = {msg: tuple(int(x) for x in reed_solomon_encode(msg, p)) 
             for msg in product(range(p), repeat=K)}

print("all messages (coeffs) -> codewords f(0...4)")
for msg, cw in codewords.items():
    print(f"   {list(msg)} -> {list(cw)}")

max_agreement = 0
msgs = list(codewords)
for i in range(len(msgs)):
    for j in range(i+1, len(msgs)):
        agree = sum(x==y for x,y in zip(codewords[msgs[i]], codewords[msgs[j]]))
        max_agreement = max(max_agreement, agree)

print(f"\nTotal distinct codewords: {len(set(codewords.values()))} (= p^K = {p**K})")
print(f"Max positions two codewords agree on: {max_agreement} (<= K-1 = {K-1})")

all messages (coeffs) -> codewords f(0...4)
   [0, 0] -> [0, 0, 0, 0, 0]
   [0, 1] -> [0, 1, 2, 3, 4]
   [0, 2] -> [0, 2, 4, 1, 3]
   [0, 3] -> [0, 3, 1, 4, 2]
   [0, 4] -> [0, 4, 3, 2, 1]
   [1, 0] -> [1, 1, 1, 1, 1]
   [1, 1] -> [1, 2, 3, 4, 0]
   [1, 2] -> [1, 3, 0, 2, 4]
   [1, 3] -> [1, 4, 2, 0, 3]
   [1, 4] -> [1, 0, 4, 3, 2]
   [2, 0] -> [2, 2, 2, 2, 2]
   [2, 1] -> [2, 3, 4, 0, 1]
   [2, 2] -> [2, 4, 1, 3, 0]
   [2, 3] -> [2, 0, 3, 1, 4]
   [2, 4] -> [2, 1, 0, 4, 3]
   [3, 0] -> [3, 3, 3, 3, 3]
   [3, 1] -> [3, 4, 0, 1, 2]
   [3, 2] -> [3, 0, 2, 4, 1]
   [3, 3] -> [3, 1, 4, 2, 0]
   [3, 4] -> [3, 2, 1, 0, 4]
   [4, 0] -> [4, 4, 4, 4, 4]
   [4, 1] -> [4, 0, 1, 2, 3]
   [4, 2] -> [4, 1, 3, 0, 2]
   [4, 3] -> [4, 2, 0, 3, 1]
   [4, 4] -> [4, 3, 2, 1, 0]

Total distinct codewords: 25 (= p^K = 25)
Max positions two codewords agree on: 1 (<= K-1 = 1)


# Step 3 Hadamard (inner code)

In [35]:
def hadamard_matrix(p):
    """Construct Hadamard matrix of order n=2^k recursively"""
    idx = np.arange(p)
    inner = np.outer(idx, idx) % p
    return phi(inner, p)

In [45]:
p = 5

idx = np.arange(p)
inner = np.outer(idx, idx) % p
print(f"f_Had exponents before phi:\n{inner}\n")
print(f"column for symbol a = 2 BEFORE phi (f_Had(2) = (2*b mod 5))")
print(inner[:,2])

#AFTER phi
H = hadamard_matrix(p)
print(f"\nHadamard matrix H = f_Had after phi:\n{H}\n")
print(f"column for symbol a = 2 AFTER phi (f_Had(2) = (phi(2*b) mod 5))")
print(H[:,2])

gram = H.conj().T @ H
print(f"H^dagger H (should be 5*I)")
print(np.round(gram.real, 3))

f_Had exponents before phi:
[[0 0 0 0 0]
 [0 1 2 3 4]
 [0 2 4 1 3]
 [0 3 1 4 2]
 [0 4 3 2 1]]

column for symbol a = 2 BEFORE phi (f_Had(2) = (2*b mod 5))
[0 2 4 1 3]

Hadamard matrix H = f_Had after phi:
[[ 1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j   ]
 [ 1.   +0.j     0.309+0.951j -0.809+0.588j -0.809-0.588j  0.309-0.951j]
 [ 1.   +0.j    -0.809+0.588j  0.309-0.951j  0.309+0.951j -0.809-0.588j]
 [ 1.   +0.j    -0.809-0.588j  0.309+0.951j  0.309-0.951j -0.809+0.588j]
 [ 1.   +0.j     0.309-0.951j -0.809-0.588j -0.809+0.588j  0.309+0.951j]]

column for symbol a = 2 AFTER phi (f_Had(2) = (phi(2*b) mod 5))
[ 1.   +0.j    -0.809+0.588j  0.309-0.951j  0.309+0.951j -0.809-0.588j]
H^dagger H (should be 5*I)
[[ 5. -0. -0. -0. -0.]
 [-0.  5.  0.  0.  0.]
 [-0.  0.  5.  0.  0.]
 [-0.  0.  0.  5.  0.]
 [-0.  0.  0.  0.  5.]]


# Step 4 Concatenate

In [47]:
def histo_encode(message , p):
    """ The histo paper encoding function (prime field, m=1)
    message: length-K list/array of field symbols in {0,...,p-1}
    returns: compelex hypervector of dimension p**2"""
    codeword = reed_solomon_encode(message, p)
    H = hadamard_matrix(p)
    blocks = H[:, codeword].T
    return blocks.reshape(-1)  #flatten to 1D array


In [56]:
p = 5
message = [2,1]
codeword = reed_solomon_encode(message, p)
hv = histo_encode(message, p)
print("message", message)
print("RS codeword", codeword)
print("hypervector_dim", hv.shape[0], "(=p^2 =", p**2, ")")
print("all entries are p-th roots of unity:",np.allclose(np.abs(hv), 1.0))


#Substitute each RS symbol c_i with its column vector:
# column i of the matrix = f_Had(c_i) = column c_i of the Hadamard table

idx = np.arange(p)
inner = np.outer(idx,idx)%p
H = hadamard_matrix(p)

print("\n column labels (RS symbols c_i):", [int(c) for c in codeword])

#INTERMEDIATE (before phi): integer f_Had columns one per RS symbol
print("\nIntermediate matrix BEFORE phi (each column = f_Had(c_i)):")
print(inner[:, codeword])

#FINAL (after phi): complex Hadamard columns, one per RS symbol
print("\nFinal matrix AFTER phi(each column = phi(f_Had(c_i)))")
print(H[:,codeword])

#The hypervector is this matrix read column by column (top to bottom , left to right)
print("\n read the matrix down each column, left to right to recover hv:",
      np.allclose(H[:,codeword].reshape(-1,order="F"),hv))


message [2, 1]
RS codeword [2 3 4 0 1]
hypervector_dim 25 (=p^2 = 25 )
all entries are p-th roots of unity: True

 column labels (RS symbols c_i): [2, 3, 4, 0, 1]

Intermediate matrix BEFORE phi (each column = f_Had(c_i)):
[[0 0 0 0 0]
 [2 3 4 0 1]
 [4 1 3 0 2]
 [1 4 2 0 3]
 [3 2 1 0 4]]

Final matrix AFTER phi(each column = phi(f_Had(c_i)))
[[ 1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j   ]
 [-0.809+0.588j -0.809-0.588j  0.309-0.951j  1.   +0.j     0.309+0.951j]
 [ 0.309-0.951j  0.309+0.951j -0.809-0.588j  1.   +0.j    -0.809+0.588j]
 [ 0.309+0.951j  0.309-0.951j -0.809+0.588j  1.   +0.j    -0.809-0.588j]
 [-0.809-0.588j -0.809+0.588j  0.309+0.951j  1.   +0.j     0.309-0.951j]]

 read the matrix down each column, left to right to recover hv: True


# Checking Quasi Orthogonality

In [59]:
p , K = 5, 2
vecs = np.stack([histo_encode(msg, p) for msg in product(range(p), repeat=K)])
vecs = (tuple(np.round(v,6) for v in vecs))
vecs = np.stack([np.array(v) for v in vecs])

norms=np.linalg.norm(vecs, axis=1)
sims = np.abs(vecs.conj() @ vecs.T) /np.outer(norms, norms)
np.fill_diagonal(sims,0.0)


print("number of distinct hypervectors = ", len(vecs))
print("max normalized similary between any two hypervectors = ", np.max(sims))
print("Proposition 1 bound (K-1)/N = (1/5) = ", (K-1)/p )


number of distinct hypervectors =  25
max normalized similary between any two hypervectors =  0.20000000816435037
Proposition 1 bound (K-1)/N = (1/5) =  0.2


# Binding is pointwise multiplication

In [69]:
p = 5
m1, m2 = [2,1], [1,3]

hv1 = histo_encode(m1, p)
hv2 = histo_encode(m2, p)

bound = hv1 * hv2 
summed = histo_encode([(a+b)%p for a, b in zip(m1, m2)], p)

print("hv1 = histo_encode(m1)")
print(hv1)
print("\nhv2 = histo_encode(m2)")
print(hv2)
print("\nbound = hv1 * hv2")
print(bound)
print("\nsummed = histo_encode(m1+m2)")
print(summed)

print("\n binding (pointwise product) equals encoding of the summed message:",
      np.allclose(bound, summed))



hv1 = histo_encode(m1)
[ 1.   +0.j    -0.809+0.588j  0.309-0.951j  0.309+0.951j -0.809-0.588j  1.   +0.j    -0.809-0.588j  0.309+0.951j
  0.309-0.951j -0.809+0.588j  1.   +0.j     0.309-0.951j -0.809-0.588j -0.809+0.588j  0.309+0.951j  1.   +0.j
  1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j     0.309+0.951j -0.809+0.588j -0.809-0.588j
  0.309-0.951j]

hv2 = histo_encode(m2)
[ 1.   +0.j     0.309+0.951j -0.809+0.588j -0.809-0.588j  0.309-0.951j  1.   +0.j     0.309-0.951j -0.809-0.588j
 -0.809+0.588j  0.309+0.951j  1.   +0.j    -0.809+0.588j  0.309-0.951j  0.309+0.951j -0.809-0.588j  1.   +0.j
  1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j     1.   +0.j    -0.809-0.588j  0.309+0.951j  0.309-0.951j
 -0.809+0.588j]

bound = hv1 * hv2
[ 1.   +0.j    -0.809-0.588j  0.309+0.951j  0.309-0.951j -0.809+0.588j  1.   +0.j    -0.809+0.588j  0.309-0.951j
  0.309+0.951j -0.809-0.588j  1.   +0.j     0.309+0.951j -0.809+0.588j -0.809-0.588j  0.309-0.951j  1.   +0.j
  1.   

# Choosing parameters for $D$ pieces of data

With $m = 1$ there are only two knobs, $p$ and $K$, and everything else follows:
$N = p$, the hypervector dimension is $p^2$, and there are $p^K$ codewords.

Three constraints pin them down.

| Constraint | Where it comes from | Requires |
|---|---|---|
| Reed–Solomon needs $N \ge K$ | $K$ coefficients evaluated at $N = p$ points | $p \ge K$ |
| Quasi-orthogonality for $D$ bundled items | Prop 1's $\mu = \frac{K-1}{N}$, with $\mu \le \frac{1}{2D-1}$ | $p \ge (K-1)(2D-1)$ |
| Enough distinct names | code size $p^K$ | $p^K \ge$ vocabulary |

Note that $D$ (how many items you superpose) and the vocabulary size (how many distinct
items *exist*) are different quantities, and they pull in opposite directions: raising $K$
buys you exponentially more names but raises $\mu$, which costs you bundling capacity.
`choose_params` takes both and returns the smallest $p^2$ that satisfies all three.


In [ ]:
def is_prime(n):
    """trial division is plenty for the p we care about"""
    if n < 2:
        return False
    for d in range(2, int(n**0.5) + 1):
        if n % d == 0:
            return False
    return True


def next_prime(n):
    """smallest prime >= n"""
    q = max(2, int(n))
    while not is_prime(q):
        q += 1
    return q


def mu(p, K):
    """Proposition 1: max normalized |<hv_i, hv_j>| over distinct codewords = (K-1)/N, N = p"""
    return (K - 1) / p


def mu_max(D):
    """quasi-orthogonality threshold that keeps D superposed items separable"""
    return 1.0 / (2 * D - 1)


def capacity(p, K):
    """how many items this (p, K) can bundle: D <= (1 + mu) / (2 mu)"""
    u = mu(p, K)
    return float("inf") if u == 0 else (1 + u) / (2 * u)


p = 5
print(f"p = {p}, K = 2  ->  mu = {mu(p, 2):.4f}, bundles up to {capacity(p, 2):.2f} items")
print(f"to bundle D = 3 you need mu <= {mu_max(3):.4f}")


In [ ]:
def choose_params(D, n_items=None, K_max=8):
    """Pick (p, K) for a code that bundles D pieces of data.

    D:       how many hypervectors get superposed into one bundle
    n_items: how many distinct things you need to be able to name (default: just D)
    K_max:   how far up to search in K

    Minimizes the hypervector dimension p^2 subject to the three constraints,
    breaking ties toward the larger K (more codewords for the same dimension).
    """
    if n_items is None:
        n_items = D
    best = None
    for K in range(1, K_max + 1):
        p = next_prime(max(2, K, (K - 1) * (2 * D - 1)))   # RS validity + Prop 1
        while p ** K < n_items:                            # enough codewords to name them
            p = next_prime(p + 1)
        cand = (p ** 2, -K, p)
        if best is None or cand < best:
            best = cand
    _, negK, p = best
    K = -negK
    return {"D": D, "n_items": n_items, "p": p, "K": K, "N": p,
            "dim": p ** 2, "n_codewords": p ** K, "d_min": p - K + 1,
            "mu": mu(p, K), "mu_max": mu_max(D), "capacity": capacity(p, K)}


par = choose_params(D=4, n_items=100)
for _k, _v in par.items():
    print(f"{_k:>12} = {_v}")


In [ ]:
print(f"{'D':>3} {'n_items':>8} | {'p':>4} {'K':>3} {'dim':>7} {'codewords':>11} "
      f"{'mu':>8} {'mu_max':>8} {'d_min':>6}")
print("-" * 74)
for _D, _n in [(1, 10), (2, 10), (2, 1000), (4, 100), (4, 100000), (8, 100), (16, 1000)]:
    r = choose_params(_D, _n)
    print(f"{r['D']:>3} {r['n_items']:>8} | {r['p']:>4} {r['K']:>3} {r['dim']:>7} "
          f"{r['n_codewords']:>11} {r['mu']:>8.4f} {r['mu_max']:>8.4f} {r['d_min']:>6}")

    assert r["p"] >= r["K"], "RS needs N >= K"
    assert r["mu"] <= r["mu_max"] + 1e-12, "fails Prop 1 quasi-orthogonality for this D"
    assert r["capacity"] >= r["D"] - 1e-9, "cannot bundle D items"
    assert r["n_codewords"] >= r["n_items"], "not enough codewords to name the vocabulary"

    # and the encoder actually accepts what we picked
    hv = histo_encode([0] * r["K"], r["p"])
    assert hv.shape == (r["dim"],), hv.shape

print("\n'dim' is the hypervector length p^2; 'codewords' = p^K is how many things you can NAME;")
print("'D' is how many you can bundle into one superposition and still pull back out;")
print("'d_min' = p - K + 1 is the RS minimum distance.")


# Encoding Real Data - Objects and Scences

In [ ]:
p , K = 13, 2

def name_to_message(i,p,K):
    """Convert integer i to a message of length K over GF(p)"""
    msg = []
    for _ in range(K):
        msg.append(i % p)
        i //= p
    return msg

#1 the codebook give every possible value its own atomic hypervector

values = ["red", "green", "blue",
          "circle", "square", "triangle",
          "small", "medium", "large"]

codebook = {name: name_to_message(i+1, p, K) for i, name in enumerate(values)}

def atom(name):
    """Look up a value name and return its atomic hypervector"""
    return histo_encode(codebook[name],p)

def encode_object(vals):
    """An Object is the binding (pointwise product) of its value atoms"""
    hv = atom(vals[0])
    for v in vals[1:]:
        hv = hv * atom(v)
    return hv

#2 build a scence
object_A = ["red", "circle", "small"]
object_B = ["blue", "square", "large"]

scene = encode_object(object_A) + encode_object(object_B)
print("object_A", object_A)
print("object_B", object_B)
print("scene hypervector dim", scene.shape[0], "(=p^2 =", p**2, ")")

object_A ['red', 'circle', 'small']
object_B ['blue', 'square', 'large']
scene hypervector dim 169 (=p^2 = 169 )


In [82]:
#3 read it back: is candidate object present in the scene? (similarity)

candidates = [["red", "circle", "small"],  #obj A
              ["blue", "square", "large"], #obj B
              ["green", "triangle", "medium"],  #absent 
              ["green", "triangle", "small"],  #absent shares small with A
              ["red", "square", "large"]] #shares value with A and B but absent


print("\nis each candidate object present in the scene? (similarity to the scene hypervector)")

for vals in candidates:
    o = encode_object(vals)
    sim = np.abs(np.vdot(o, scene))/(np.linalg.norm(o) * np.linalg.norm(scene))
    verdict = "PRESENT" if sim > 0.3 else "ABSENT"
    print(f" {str(vals):34s} -> similarity = {sim: .2f} -> {verdict}")


is each candidate object present in the scene? (similarity to the scene hypervector)
 ['red', 'circle', 'small']         -> similarity =  0.71 -> PRESENT
 ['blue', 'square', 'large']        -> similarity =  0.71 -> PRESENT
 ['green', 'triangle', 'medium']    -> similarity =  0.00 -> ABSENT
 ['green', 'triangle', 'small']     -> similarity =  0.00 -> ABSENT
 ['red', 'square', 'large']         -> similarity =  0.00 -> ABSENT


# Choosing Parameters for D pieces of data

In [ ]:
from sympy import nextprime

def choose_parameters(D)aas